In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

---

## Env

In [ ]:
import os
from dotenv import load_dotenv
from src.utils.api_key_utils import api_key_from_file

load_dotenv()

# Optional
WANDB_API_KEY = api_key_from_file("../api_keys/WANDB_KEY.txt")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

# Optional
OPENPIPE_API_KEY = api_key_from_file("../api_keys/OPENPIPE_KEY.txt")
if OPENPIPE_API_KEY:
    os.environ["OPENPIPE_API_KEY"] = OPENPIPE_API_KEY
    

# TODO: check env vars
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"  # Set OMP_NUM_THREADS to 1 to avoid multithreading issues with vLLM
os.environ["NCCL_CUMEM_ENABLE"] = "1"  # Enable NCC

## Backend

In [ ]:
import art

from openpipe.client import OpenPipe
from art.local import LocalBackend

op_client = OpenPipe()
print("OpenPipe client initialized")

backend = LocalBackend(in_process=True)
print("Local backend initialized")

## Model

In [ ]:
from src.configs.art_model_config import configs as art_model_configs
from src.configs.vllm_model_config import model_configs as vllm_model_configs

print("Available ART model configurations:")
print(art_model_configs.keys())

model_name = "unsloth/Qwen2.5-14B-Instruct"

art_config = art_model_configs[model_name]
vllm_config = vllm_model_configs[model_name]

In [ ]:
from art.utils import output_dirs
from datetime import datetime

class DirConfig:
    def __init__(
        self,
        model_name: str,
        project_name: str,
        art_path: str,
    ) -> None:
        self.model_name = model_name
        self.project_name = project_name 
        self.art_path = art_path
        self.run_name = self._generate_run_name(model_name)

    @property
    def model_output_dir(self) -> str:
        """
        Get the output directory for the model.
        """
        return output_dirs.get_output_dir_from_model_properties(
            project=self.project_name,
            name=self.model_name,
            art_path=self.art_path,
        )

    @property
    def trajectories_dir(self) -> str:
        return output_dirs.get_trajectories_dir(
            model_output_dir=self.model_output_dir,
        )

    def _generate_run_name(self, model_name: str) -> str:
        """
        Generate a run name based on the model name and current date.
        """
        model_name = self.model_name.split("/")[-1]
        date_str = datetime.now().strftime("%m_%d_%H_%M")
        return f"{model_name}_{date_str}"

    def get_step_checkpoint_dir(self, step: int) -> str:
        """
        Get the checkpoint directory for a specific step.
        """
        return output_dirs.get_step_checkpoint_dir(model_output_dir=self.model_output_dir, step=step)


In [ ]:
from art.dev.model import InitArgs, EngineArgs, PeftArgs, TrainerArgs, InternalModelConfig

dir_config = DirConfig(
    model_name=model_name,
    project_name="easy2hard_dac_v2",
    art_path=backend._path,
)

model = art.TrainableModel(
    name=dir_config.run_name,
    project=dir_config.project_name,
    base_model=dir_config.model_name,
    _internal_config=art_config,
)

await model.register(backend)

## Data

In [ ]:
from datasets import Dataset, load_dataset, DatasetDict

dataset_dict: DatasetDict = load_dataset(
    path="furonghuang-lab/Easy2Hard-Bench",
    name="E2H-AMC",
    split=None,
)  # type: ignore

train_data: Dataset = dataset_dict["train"]
test_data: Dataset = dataset_dict["eval"]

## Training

In [ ]:
from dataclasses import dataclass, field
import art

@dataclass
class TrainingConfig:
    epochs: int = 10
    num_groups: int = 2
    group_rollouts: int = 10
    verbose: bool = False
    
    config: art.types.TrainConfig = field(default_factory=art.types.TrainConfig)
    _config: art.dev.train.TrainConfig | None = None
    
    
train_config = TrainingConfig(
    epochs=10,
    num_groups=2,
    group_rollouts=10,
    config=art.types.TrainConfig(learning_rate=1e-5)
)

### Inference Clients

In [ ]:
from src.vllm_client import VllmClient, VllmRouter, ArtVLLMClient

inference_clients: list[VllmClient] = [ArtVLLMClient(model)]

vllm_server_ports = []
for port in vllm_server_ports:
    vllm_client = VllmClient(port=port, base_model=model_name)
    inference_clients.append(vllm_client)

vllm_router: VllmRouter = VllmRouter(vllm_clients=inference_clients)

### Inference Test

In [ ]:
from importlib import reload
import src.utils.visualize
import src.dac_agent
reload(src.utils.visualize)
reload(src.dac_agent)

In [ ]:
import random
from datasets import Dataset
from src.dac_agent import AgentNode, SysPrompt, StopCriteria, ChatMessage
from src.configs.sys_prompt import DaCSystemPrompt

sys_prompt = SysPrompt(
    root_prompt=DaCSystemPrompt.dac_sys_prompt_v2_3,
    inter_prompt=DaCSystemPrompt.dac_sys_prompt_v2_3,
    leaf_prompt=DaCSystemPrompt.dac_sys_prompt_v2_3_leaf,
)

stop_criteria = StopCriteria(
    max_depth=1,
    max_tasks=5,
    max_rounds=5,
)

vllm_client = vllm_router.__next__()

agent = AgentNode(
    client=vllm_client.client,
    model=vllm_client.get_inference_name(),
    system_prompt=sys_prompt,
    stop_criteria=stop_criteria,
)

idx = random.randint(0, len(train_data) - 1)
sample = train_data[idx]
problem = sample["problem"]
answer = sample["answer"]

print(f"Problem: {problem}")
print(f"Answer: {answer}")
print("-" * 200)
print()

message = ChatMessage(role="user", content=problem)
await agent.answer(message, verbose=True)

### General Trainer

In [ ]:
from abc import abstractmethod

from art.dev.model import InternalModelConfig


class Trainer:
    def __init__(
        self,
        model: art.TrainableModel,
        inference_clients: list[VllmClient],
        dir_config: DirConfig,
    ):
        self.model = model
        self.dir_config = dir_config
        self.vllm_router = VllmRouter(vllm_clients=inference_clients)

    @property
    def model_config(self) -> InternalModelConfig:
        config = self.model._internal_config
        if config is None:
            raise ValueError("Model configuration is not set.")
        return config

    def process_dataset(self, dataset: Dataset) -> Dataset:
        """
        Process the dataset before training.
        This method should be implemented by subclasses to handle specific dataset processing.
        """
        return dataset

    def reload_lora(self, step: int) -> bool:
        prev_checkpoint_dir = None
        curr_checkpoint_dir = None

        if step > 0:
            curr_checkpoint_dir = self.dir_config.get_step_checkpoint_dir(step)

        if step - 1 > 0:
            prev_checkpoint_dir = self.dir_config.get_step_checkpoint_dir(step - 1)

        if prev_checkpoint_dir is not None:
            print("Unloading lora")
            if not vllm_router.unload_lora(dir_config.run_name):
                print(f"Failed to unload lora from {prev_checkpoint_dir}")
                return False

        if curr_checkpoint_dir is not None:
            print(f"Loading lora from {curr_checkpoint_dir}")
            if not vllm_router.load_lora(dir_config.run_name, curr_checkpoint_dir):
                print(f"Failed to load lora from {curr_checkpoint_dir}")
                return False

        return True

    async def fit(
        self,
        train_data: Dataset,
        val_data: Dataset,
        train_config: TrainingConfig,
    ):
        model = self.model
        vllm_router = self.vllm_router

        # Unload all lora adapters before starting the training
        vllm_router.unload_all_loras()

        for i in range(await model.get_step(), train_config.epochs):
            self.reload_lora(i)

            epoch_data = train_data.shuffle()
            epoch_data = epoch_data.take(train_config.num_groups * train_config.group_rollouts)
            epoch_data = self.process_dataset(epoch_data)

            group_data = [
                epoch_data.shard(num_shards=train_config.num_groups, index=i, contiguous=True)
                for i in range(train_config.num_groups)
            ]

            train_groups = [
                art.TrajectoryGroup([self.rollout(sample, vllm_router.__next__()) for sample in data])
                for data in group_data
            ]

            trajectory_groups = await art.gather_trajectory_groups(train_groups, pbar_desc="gather")

            await model.delete_checkpoints()

            await model.train(
                trajectory_groups,
                config=train_config.config,
                _config=train_config._config,
                verbose=train_config.verbose,
            )

        return self.model

    @abstractmethod
    async def rollout(
        self,
        sample: dict,
        vllm_client: VllmClient,
    ) -> art.Trajectory | Exception:
        raise NotImplementedError("Subclasses must implement the rollout method.")

### Rewards

In [ ]:
from src.utils.text_utils import extract_text_between_markers
from src.configs.markers import Markers
from src.dac_agent import ChatMessage

def answer_reward(sample: dict[str, str], message: ChatMessage) -> float:
    role = message.role
    content = message.content

    if role != "assistant":
        raise ValueError(f"Expected role 'assistant', got '{role}'")

    prompt = sample["problem"]
    answer = sample["answer"]
    total_reward = 0.0

    answer_list = extract_text_between_markers(content, Markers.ANSWER_START, Markers.ANSWER_END)

    if len(answer_list) == 0:
        total_reward -= 3
        llm_answer = content.strip()

    elif len(answer_list) > 1:
        total_reward -= 0.5 * (len(answer_list) - 1)
        llm_answer = answer_list[-1].strip()

    else:
        llm_answer = answer_list[-1].strip()

    if llm_answer.lower().strip() == answer.lower().strip():
        total_reward += 1.5
    else:
        total_reward -= 1

    return total_reward


def format_reward(message: ChatMessage) -> float:
    role = message.role
    response = message.content

    if role != "assistant":
        raise ValueError(f"Expected role 'assistant', got '{role}'")

    reward = 0.0

    tasks = extract_text_between_markers(response, Markers.TASK_START, Markers.TASK_END)
    answers = extract_text_between_markers(response, Markers.ANSWER_START, Markers.ANSWER_END)

    if len(tasks) == 0 and len(answers) == 0:
        reward -= 0.1  # Penalize for no tasks or answers
    elif len(tasks) > 0 and len(answers) == 0:
        reward += 0.2  # ** len(tasks)  # Reward for tasks without answers
    elif len(tasks) == 0 and len(answers) > 0:
        reward += 0.2 ** len(answers)  # Reward for answers without tasks, diminishing with more answers
    else:
        reward -= 0.1 ** (
            1 / min(len(tasks), len(answers))
        )  # Penalize for each task that was also answered by the agent

    tasks_diff = abs(response.count(Markers.TASK_START) - response.count(Markers.TASK_END))
    answers_diff = abs(response.count(Markers.ANSWER_START) - response.count(Markers.ANSWER_END))

    if tasks_diff > 0:
        reward -= 0.1 ** (1 / tasks_diff)
    if answers_diff > 0:
        reward -= 0.1 ** (1 / answers_diff)

    return reward

### Trainer Implementation

In [ ]:
from datasets import Dataset
from src.dac_agent import AgentNode, SysPrompt, StopCriteria, ChatMessage
from src.configs.sys_prompt import DaCSystemPrompt
from openai.types.chat.chat_completion import Choice


sys_prompt = SysPrompt(
    root_prompt=DaCSystemPrompt.dac_sys_prompt_v2_3,
    inter_prompt=DaCSystemPrompt.dac_sys_prompt_v2_3,
    leaf_prompt=DaCSystemPrompt.dac_sys_prompt_v2_3_leaf,
)

stop_criteria = StopCriteria(
    max_depth=1,
    max_tasks=5,
    max_rounds=5,
)

class Easy2HardTrainer(Trainer):
    def process_dataset(self, dataset: Dataset) -> Dataset:
        return dataset.sort("item_difficulty", reverse=False)

    async def rollout(
        self,
        sample: dict,
        vllm_client: VllmClient,
    ) -> art.Trajectory | Exception:
        """
        Rollout method to be implemented by subclasses.
        This method should handle the interaction with the VLLM client and return a trajectory.
        """

        question = sample["problem"]
        answer = sample["answer"]

        agent = AgentNode(
            client=vllm_client.client,
            model=vllm_client.get_inference_name(),
            system_prompt=sys_prompt,
            stop_criteria=stop_criteria,
        )

        message = ChatMessage(role="user", content=question)
        try:
            max_tokens = self.model_config["max_seq_length"] if "max_seq_length" in self.model_config else None
            trajectory = await agent.chat(message, max_tokens=max_tokens)
        except Exception as e:
            print("caught exception generating chat completion")
            print(e)
            raise e
        
        ans_message = ChatMessage.model_validate(trajectory.messages()[-1], from_attributes=True)
        ans_reward = answer_reward(sample, ans_message)
        trajectory.reward += ans_reward
        
        for item in trajectory.messages_and_choices:
            if isinstance(item, Choice):
                msg = ChatMessage.model_validate(item.message, from_attributes=True)
                trajectory.reward += format_reward(msg)

        agent_answer = agent._parse_answer(ans_message)

        # Add the answer and agent_answer to the trajectory metrics
        trajectory.metadata["answer"] = answer
        trajectory.metadata["agent_answer"] = agent_answer.content
        trajectory.metadata["item_difficulty"] = sample["item_difficulty"]
        trajectory.metadata["contest"] = sample["contest"]

        return trajectory

In [ ]:
trainer = Easy2HardTrainer(
    model=model,
    inference_clients=inference_clients,
    dir_config=dir_config,
)

await trainer.fit(
    train_data=train_data,
    val_data=test_data,
    train_config=train_config,
)